**Дистилляция 5-фолдового ансамбля nnU-Net ResEnc M → одна модель**

Учитель: 5 фолдов (fp16, cuda:1) → средний softmax. Студент: same-arch, тёплый старт из fold_0.

Лосс: `Σᵢ wᵢ · [W_DICE · softDice + (1−W_DICE) · softCE]`, wᵢ = 1/2ⁱ по уровням deep supervision.

**Протокол экспериментов** (одна переменная за прогон, сплит фиксирован seed'ом):
| Дата | База (epoch) | LR | W_DICE | HARD_GT | DA5 | OVERSAMPLE | Результат (честный Dice) |
|------|--------------|-----|--------|---------|-----|------------|--------------------------|
| ...  | fresh        | .01 | .5     | 0       | 0   | .33        | ... |


Правила: (1) `DISTILL_EPOCHS` не менять внутри resume-цепочки — от него зависит poly-LR;
(2) чистая абляция = свежий старт (`REQUIRE_CKPT=0`), различайте прогоны по version notes в Kaggle;
(3) сравнивать только на внешнем холдауте, не по pseudo-dice.

Как проводить эксперименты (короткая памятка)
Боевой resume-прогон: все ⚙ по умолчанию, только следите за [resume], [lr] peak LR = 0.01 и [DA] effective mirror_axes: None в логе.
Абляция (после завершения 200 эпох): REQUIRE_CKPT=0 + свежий старт, меняете один параметр, заполняете строку таблицы в markdown-ячейке. Кандидаты по порядку: HARD_GT=0.25 → USE_DA5=1 → PEAK_LR=0.006 → W_DICE=0.3 (границы) или 0.7 (объёмы) → OVERSAMPLE=0.45.
Сравнение прогонов — только по perform_actual_validation на общем холдауте (ячейка 13), не по pseudo-dice.
Два напоминания: токен вставляйте через Secrets, не в код; и если [resume]-проверка в ячейке 11 покажет current_epoch порядка 1000 — fallback взял чекпоинт учителя, гвард в конце ячейки остановит обучение, но разбирайтесь с input-ами до запуска.

In [ ]:
# ЯЧЕЙКА 1 — ПУЛЬТ УПРАВЛЕНИЯ (все экспериментальные параметры)
# ⚙ = можно/нужно крутить. Меняйте ОДИН параметр за прогон.
# ============================================================
import os as _os

# ---------- инфраструктура (не трогать без нужды) ----------
_os.environ.setdefault("DISTILL_BUDGET_H", "11.3")   # мягкий бюджет сессии, ч (чистый выход до 12ч-лимита)
_os.environ.setdefault("DISTILL_REQUIRE_CKPT", "1")  # запрет старта без чекпоинта (в SMOKE/CPU отключается сам)
# _os.environ["KAGGLE_BEARER"] = "KGAT_..."          # ⚠️ НЕ вшивать в код! Вставляйте через Kaggle Secrets (Add-ons)

# ---------- экспериментальные ----------
_os.environ.setdefault("DISTILL_EPOCHS", "200")      # ⚙ длина poly-цикла. НЕ менять при resume (сдвинет LR-кривую)
_os.environ.setdefault("DISTILL_LR", "0.01")         # ⚙ peak LR. С тёплым стартом/на поздних стадиях: 0.005–0.007
                                                     #    (меньше деструкции тёплых весов). Работает и на resume.
_os.environ.setdefault("DISTILL_W_DICE", "0.5")      # ⚙ НОВОЕ: баланс soft Dice : soft CE (0.5 = как раньше 1:1)
                                                     #    → 1.0 только объёмная согласованность (крупные структуры)
                                                     #    → 0.0 только по-воксельный сигнал (границы, мелкие структуры)
_os.environ.setdefault("DISTILL_HARD_GT", "0.0")     # ⚙ вес hard-GT лосса. Сетка Fast-nnUNet: 0.1 / 0.2 / 0.3 / 0.5
                                                     #    0.2–0.3 — кандидат №1: коррекция ошибок ансамбля (Cochlea/Lens)
_os.environ.setdefault("DISTILL_USE_DA5", "0")       # ⚙ НОВОЕ: сильная аугментация DA5 из nnunetv2. Вы в целевой зоне
                                                     #    (<100 кейсов); +10–15% времени эпохи. Кандидат №2.
_os.environ.setdefault("DISTILL_OVERSAMPLE", "0.33") # ⚙ НОВОЕ: доля патчей с форграундом (дефолт nnU-Net 0.33)
                                                     #    0.40–0.50 — попытка помочь Cochlea/Lens/нервам; риск дисбаланса
_os.environ.setdefault("DISTILL_INIT", "1")          # тёплый старт из fold_0 (нужен только для старта с нуля)
_os.environ.setdefault("DISTILL_SMOKE", "0")         # смоук: 1 эпоха × 30 итераций, 8 случаев
_os.environ.setdefault("DISTILL_CPU", "0")           # CPU_MODE: прогон пайплайна без GPU (учителя fp32 на CPU)
_os.environ.setdefault("UPLOAD_EVERY", "5")          # ⚙ аплоад чекпоинта каждые N эпох (окно потери ≈ N·12.5 мин)

# Пульт управления: все переменные дистилляции

Каждая переменная — что делает, к чему приводит включение/выключение или диапазон.
Меняйте **ОДИН параметр за прогон** (иначе не поймёте причину изменения результата).

| Переменная | Что это | Дефолт (рецепт) | Диапазон / альтернатива | Комментарий |
|---|---|---|---|---|
| `DISTILL_EPOCHS` | Длина цикла обучения и poly-LR распада | 200 | 250+ дольше, выигрыш не доказан | **Не менять внутри resume-цепочки** — сдвинет LR-кривую |
| `DISTILL_LR` | Peak LR оптимизатора SGD+PolyLR | 0.01 (дефолт nnU-Net) | 0.005–0.007 — бережнее к тёплым весам | После resume LR идёт по НОВОЙ poly-кривой от этого значения |
| `DISTILL_W_DICE` | Баланс soft-Dice : soft-CE в лоссе | 0.5 (поровну) | →1.0 объёмы/крупные органы; →0.0 границы/мелкие структуры | Главный рычаг акцента «объёмы vs границы» |
| `DISTILL_HARD_GT` | Вес жёстких GT-масок: итог = (1−w)·soft + w·hard | 0 (чистая дистилляция) | 0.2–0.3 коррекция ошибок ансамбля на малых структурах; 0.5+ риск потери мягкости | Кандидат №1 для абляций |
| `DISTILL_USE_DA5` | Сильная аугментация DA5 (усиленные геометрия+интенсивность) | 0 (стандартные) | 1 — +робастность при <100 кейсах (наша зона); +10–15% времени эпохи | Кандидат №2; TTA в проде нет — совместимо |
| `DISTILL_OVERSAMPLE` | Доля патчей с форграундом (органом) в батче | 0.33 (дефолт nnU-Net) | 0.4–0.5 больше мелких органов (Cochlea/Lens/нервам); риск дисбаланса | Через oversample_foreground_percent |
| `DISTILL_INIT` | Тёплый старт студента весами fold_0 учителя | 1 (сходимость в разы быстрее) | 0 — холодный старт, дольше, но чище абляция | |
| `DISTILL_BUDGET_H` | Мягкий бюджет сессии: за час до 12ч-лимита чистое завершение | 11.3 (не трогать) | меньше — короче сессии, чаще переносы | Kaggle НЕ сохраняет working при жёстком обрыве |
| `DISTILL_REQUIRE_CKPT` | Запрет старта без чекпоинта продолжения | 1 (боевой resume) | 0 — старт с нуля (для абляций) | В SMOKE/CPU отключается автоматически |
| `DISTILL_SMOKE` | Смоук: 1 эпоха × 30 итераций × 8 кейсов | 0 | 1 — проверка конвейера ~10 мин GPU | «Training done» в CPU-смоуке может скрывать падение валидации — смотреть val_loss |
| `DISTILL_CPU` | Весь конвейер на CPU fp32 | 0 (GPU) | 1 — проверка без квоты (медленно, только смоук) | |
| `UPLOAD_EVERY` | Раз в N эпох дублировать чекпоинт в приватный датасет | 5 (≈раз в час; окно потери = N×12.5 мин) | больше — длиннее окно потери | Требует KAGGLE_BEARER |
| `KAGGLE_BEARER` | KGAT-токен аплоада через REST | — (Secrets!) | — | **Только Add-ons → Secrets, никогда в коде** (история версий уйдёт на GitHub) |

Зафиксированы в коде (менять не рекомендую): веса уровней deep supervision `1/2ⁱ`;
seed сплита `20261003` (сравнимость прогонов); `batch=2`, `patch=56×192×192` — из plans.json учителя.

In [ ]:
# ЯЧЕЙКА 2 — установка, импорты, производные флаги
# ============================================================
import subprocess, sys, os
try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

def _pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

try:
    import nnunetv2  # noqa: F401
    print(f"[setup] nnunetv2 {nnunetv2.__version__}")
except ImportError:
    print("[setup] ставлю nnunetv2==2.8.1 ...")
    # 2.8.1: новая сигнатура build_network_architecture и MetaLogger —
# оба варианта поддержаны ниже (версия-агностичный код)
_pip("nnunetv2==2.8.1")

import numpy as np  # noqa: E402
import torch  # noqa: E402

SMOKE = os.environ.get("DISTILL_SMOKE") == "1"
CPU_MODE = os.environ.get("DISTILL_CPU") == "1"
NUM_EPOCHS = int(os.environ["DISTILL_EPOCHS"])
PEAK_LR = float(os.environ["DISTILL_LR"])               # ⚙ peak LR poly-расписания
W_DICE = float(os.environ["DISTILL_W_DICE"])            # ⚙ баланс softDice/softCE
HARD_GT_WEIGHT = float(os.environ["DISTILL_HARD_GT"])   # ⚙ вес hard-GT
USE_DA5 = os.environ.get("DISTILL_USE_DA5") == "1"      # ⚙ DA5
OVERSAMPLE = float(os.environ["DISTILL_OVERSAMPLE"])    # ⚙ форграунд-оверсэмплинг
BUDGET_H = float(os.environ["DISTILL_BUDGET_H"])
UPLOAD_EVERY = int(os.environ["UPLOAD_EVERY"])
# REQUIRE_CKPT осмыслен только для боевого GPU-прогона:
REQUIRE_CKPT = (os.environ.get("DISTILL_REQUIRE_CKPT") == "1") and not SMOKE and not CPU_MODE

print(f"[cfg] epochs={NUM_EPOCHS} lr={PEAK_LR} w_dice={W_DICE} hard_gt={HARD_GT_WEIGHT} "
      f"da5={USE_DA5} oversample={OVERSAMPLE} budget={BUDGET_H}h smoke={SMOKE} cpu={CPU_MODE}")

In [ ]:
# ЯЧЕЙКА 3 — поиск входов (учитель + preprocessed)
# Фильтр учителя: у кандидата должны лежать ВСЕ 5 checkpoint_final.
# ============================================================
import glob

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

def _is_teacher_results(d):
    return all(os.path.isfile(os.path.join(d, f"fold_{i}", "checkpoint_final.pth"))
               for i in range(5))

in_pp = sorted(glob.glob("/kaggle/input/**/nnUNet_preprocessed/Dataset*", recursive=True))
IN_PREPROCESSED = next((h for h in in_pp if glob.glob(os.path.join(h, "*", "*.b2nd"))),
                       in_pp[0] if in_pp else None)
in_res = sorted(glob.glob("/kaggle/input/**/nnUNet_results/Dataset*/*__*__3d_fullres", recursive=True))
IN_RESULTS = next((h for h in in_res if _is_teacher_results(h)), None)
assert IN_PREPROCESSED, "nnUNet_preprocessed не найден в /kaggle/input"
assert IN_RESULTS, "nnUNet_results учителя не найден в /kaggle/input"
print(f"[setup] preprocessed: {IN_PREPROCESSED}")
print(f"[setup] teacher results: {IN_RESULTS}")

In [ ]:
# ЯЧЕЙКА 4 — preprocessed в /kaggle/tmp, переменные окружения
# PP в /kaggle/tmp (НЕ в working): output версии остаётся ~2-3 ГБ
# (только чекпоинты), симлинк не материализуется при Save Version.
# ============================================================
import shutil, time

PP = "/kaggle/tmp/nnUNet_preprocessed"
DATASET_DIR = os.path.join(PP, os.path.basename(IN_PREPROCESSED.rstrip("/")))
if not os.path.isdir(DATASET_DIR):
    print("[setup] копирую preprocessed в tmp ...")
    t0 = time.time()
    shutil.copytree(IN_PREPROCESSED, DATASET_DIR)
    print(f"[setup] скопировано за {time.time() - t0:.0f} с")

os.environ["nnUNet_preprocessed"] = PP
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"
os.environ.setdefault("nnUNet_n_proc_DA", "4")
os.makedirs(os.environ["nnUNet_results"], exist_ok=True)
os.makedirs(os.environ["nnUNet_raw"], exist_ok=True)

In [ ]:
# ЯЧЕЙКА 5 — сплит 85/15 (детерминированный, seed фиксирован)
# ⚙ МЕНЯТЬ СИД НЕ НУЖНО: одинаковый сплит = сравнимость прогонов.
# Если добавите новые кейсы — сплит пересоберётся и val поплывёт:
# тогда фиксируйте val по списку имён (или делите группы пациентов).
# ============================================================
from batchgenerators.utilities.file_and_folder_operations import join, load_json  # noqa: E402
# ВАЖНО: импорты nnunetv2-training — ПОСЛЕ установки env в ячейке 4

SPLITS_PATH = os.path.join(DATASET_DIR, "splits_final.json")
real_dirs = [d for d in glob.glob(os.path.join(DATASET_DIR, "*_3d_fullres"))
             if os.path.isdir(d) and os.listdir(d)]
assert real_dirs, f"нет папки данных *_3d_fullres в {DATASET_DIR}"
real_data_dir = real_dirs[0]
expected_data_dir = os.path.join(DATASET_DIR, "nnUNetResEncUNetMPlans_3d_fullres")
if os.path.realpath(expected_data_dir) != os.path.realpath(real_data_dir) \
        and not os.path.exists(expected_data_dir):
    os.symlink(real_data_dir, expected_data_dir)
DATA_DIR = real_data_dir
print(f"[setup] папка данных: {DATA_DIR}")

cases = sorted(os.path.splitext(f)[0] for f in os.listdir(DATA_DIR)
               if f.endswith(".b2nd") and not f.endswith("_seg.b2nd"))
if not cases:
    cases = sorted(os.path.splitext(f)[0] for f in os.listdir(DATA_DIR)
                   if f.endswith(".npy") and not f.endswith("_seg.npy"))
if not cases:
    cases = sorted(os.path.splitext(os.path.splitext(f)[0])[0]
                   for f in os.listdir(DATA_DIR) if f.endswith(".npz"))
assert cases, f"нет данных в {DATA_DIR}"
rng = np.random.RandomState(20261003)                 # ⚙ seed сплита — фиксирован для сравнимости
perm = rng.permutation(len(cases))
n_val = max(4, int(round(len(cases) * 0.15)))
val_ids = [cases[i] for i in perm[:n_val]]
train_ids = [cases[i] for i in perm[n_val:]]
if SMOKE:
    train_ids = train_ids[:8]
with open(SPLITS_PATH, "w") as f:
    import json as _json
    _json.dump([{"train": train_ids, "val": val_ids}], f)
print(f"[setup] сплит: train {len(train_ids)}, val {len(val_ids)}")

In [ ]:
# ЯЧЕЙКА 6 — планы, dataset.json, фабрика сети
# ============================================================
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer  # noqa: E402
from nnunetv2.training.nnUNetTrainer.variants.data_augmentation.nnUNetTrainerNoMirroring import (  # noqa: E402
    nnUNetTrainerNoMirroring)
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager  # noqa: E402

PLANS_FILE = join(DATASET_DIR, "nnUNetResEncUNetMPlans.json")
if not os.path.isfile(PLANS_FILE):
    cand = glob.glob(join(DATASET_DIR, "*Plans*.json"))
    assert cand, f"plans не найден в {DATASET_DIR}"
    PLANS_FILE = cand[0]
plans = load_json(PLANS_FILE)

DATASET_JSON = join(DATASET_DIR, "dataset.json")
if not os.path.isfile(DATASET_JSON):
    organ_labels = ["BrachialPlexus_R", "BrachialPlexus_L", "SpinalCanal", "SpinalCord",
                    "Trachea", "Esophagus", "Thyroid", "A_Carotid_R", "A_Carotid_L",
                    "Larynx", "PharynxConst_I", "PharynxConst_M", "PharynxConst_S",
                    "OpticNerve_R", "OpticNerve_L", "Brain", "Submandibular_R",
                    "Submandibular_L", "Parotid_R", "Parotid_L", "Lips", "Tongue",
                    "Mandible", "Eye_R", "Lens_R", "Eye_L", "Lens_L", "TMJoint_R",
                    "TMJoint_L", "Cochlea_R", "Cochlea_L"]
    labels = {"background": 0}
    labels.update({n: i + 1 for i, n in enumerate(organ_labels)})
    dj = {"channel_names": {"0": "ct"}, "labels": labels,
          "numTraining": len(cases), "file_ending": ".nii.gz"}
    with open(DATASET_JSON, "w") as f:
        json.dump(dj, f, indent=2)
    print("[setup] dataset.json восстановлен")
dataset_json = load_json(DATASET_JSON)

CONFIG = "3d_fullres"
pm = PlansManager(plans)
cm = pm.get_configuration(CONFIG)

def build_net():
    from nnunetv2.utilities.label_handling.label_handling import determine_num_input_channels
    label_manager = pm.get_label_manager(dataset_json)
    num_in = determine_num_input_channels(pm, cm, dataset_json)
    try:
        # nnunetv2 >= 2.7: принимает plans/configuration managers напрямую
        return nnUNetTrainer.build_network_architecture(
            pm, cm, num_in, label_manager.num_segmentation_heads, True)
    except TypeError:
        # nnunetv2 2.6.x: имя класса архитектуры + kwargs из plans
        return nnUNetTrainer.build_network_architecture(
            cm.network_arch_class_name, cm.network_arch_init_kwargs,
            cm.network_arch_init_kwargs_req_import,
            num_in, label_manager.num_segmentation_heads, True)

In [ ]:
# ЯЧЕЙКА 7 — загрузка 5 учителей (fp16 на cuda:1; fp32/CPU в CPU_MODE)
# ============================================================
print("[teachers] загрузка 5 фолдов учителя...")
t0 = time.time()
teacher_nets = []
teacher_device = "cpu" if CPU_MODE else ("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
print(f"[teachers] устройство учителей: {teacher_device}")
for fold in range(5):
    sd = torch.load(join(IN_RESULTS, f"fold_{fold}", "checkpoint_final.pth"),
                    map_location="cpu", weights_only=False)
    net = build_net()
    net.load_state_dict(sd["network_weights"])
    net.eval()
    net.to(teacher_device)
    if not CPU_MODE:
        net.half()
    for p in net.parameters():
        p.requires_grad_(False)
    teacher_nets.append(net)
    del sd
    print(f"[teachers] fold {fold}: ok ({time.time() - t0:.0f} с)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ЯЧЕЙКА 8 — софт-таргеты учителя и функции лосса
#
# Итоговая формула (на каждом уровне i deep supervision, wᵢ = 1/2ⁱ):
#   L = Σᵢ wᵢ · [ W_DICE·softDice(sᵢ, p̄ₜ) + (1−W_DICE)·softCE(sᵢ, p̄ₜ) ]
# W_DICE → 1.0: акцент на объёмы (крупные структуры)
# W_DICE → 0.0: акцент на воксельный сигнал (границы, мелкие структуры)
# p̄ₜ = средний softmax 5 фолдов (конвенция nnU-Net; T=1 неявно).
# ============================================================
@torch.no_grad()
def teacher_probs(data):
    acc = None
    d = data.to(teacher_device, non_blocking=True)
    if not CPU_MODE:
        d = d.half()
    for net in teacher_nets:
        logits = net(d)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        p = torch.softmax(logits.float(), dim=1)
        acc = p if acc is None else acc + p
    return (acc / len(teacher_nets)).detach().to(data.device)

def soft_targets_for_ds(t_probs, out_shapes):
    outs = []
    for shp in out_shapes:
        if tuple(t_probs.shape[2:]) == tuple(shp):
            outs.append(t_probs)
        else:
            outs.append(torch.nn.functional.interpolate(
                t_probs, size=shp, mode="area").clamp_min(1e-6))
    return outs

def soft_ce(s_logits, t_probs):
    logp = torch.log_softmax(s_logits.float(), dim=1)
    return -(t_probs * logp).sum(dim=1).mean()   # сумма по классам, mean по вокселям (каноника)

def soft_dice(s_logits, t_probs, eps=1e-6):
    s = torch.softmax(s_logits.float(), dim=1)
    dims = (0, 2, 3, 4)
    inter = (s * t_probs).sum(dims)
    denom = s.sum(dims) + t_probs.sum(dims)
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()

In [ ]:
# ЯЧЕЙКА 9 — страховочный аплоад чекпоинта в приватный датасет
# ============================================================
class _SessionOver(Exception):
    """Мягкое завершение по бюджету: чекпоинт уже сохранён."""

def kaggle_ckpt_upload(path, notes):
    try:
        import requests
        tok = os.environ.get("KAGGLE_BEARER")
        if not tok:
            print("[ckpt-upload] нет KAGGLE_BEARER — пропуск")
            return
        H = {"Authorization": f"Bearer {tok}"}
        base = "https://www.kaggle.com/api/v1"
        fname = os.path.basename(path)
        size = os.path.getsize(path)
        mtime = int(os.path.getmtime(path) * 1000)   # МИЛЛИСЕКУНДЫ (API Kaggle ожидает мс)
        r = requests.post(f"{base}/datasets/upload/file/{size}/{mtime}",
                          params={"fileName": fname}, headers=H, timeout=300)
        r.raise_for_status()
        info = r.json()
        with open(path, "rb") as f:
            pr = requests.put(info["createUrl"], data=f, timeout=900)  # НЕ 7200: зависший PUT съест окно
        pr.raise_for_status()
        owner, slug = "iliaszalialov", "nnunet-distill-ckpt"
        vbody = {"versionNotes": notes, "files": [{"token": info["token"]}], "convertToCsv": False}
        r = requests.post(f"{base}/datasets/create/version/{owner}/{slug}",
                          json=vbody, headers=H, timeout=300)
        if r.status_code == 404:
            cbody = {"title": "nnunet distill ckpt", "slug": slug, "ownerSlug": owner,
                     "licenseName": "CC0-1.0", "isPrivate": True,
                     "files": vbody["files"], "convertToCsv": False}
            r = requests.post(f"{base}/datasets/create/new", json=cbody, headers=H, timeout=300)
        r.raise_for_status()
        print(f"[ckpt-upload] ок ({notes})")
    except Exception as e:
        body = getattr(getattr(e, "response", None), "text", "")
        print(f"[ckpt-upload] ОШИБКА (обучение продолжается): {e} {body[:300]}")

In [ ]:
# ЯЧЕЙКА 10 — трейнер дистилляции (+ DA5 через MRO)
# Класс называется nnUNetTrainerDistill в ОБЕИХ ветках → одна и та же
# папка output и один resume-глоб независимо от DA5.
# ============================================================
_DA5 = None
if USE_DA5:
    try:
        from nnunetv2.training.nnUNetTrainer.variants.data_augmentation.nnUNetTrainerDA5 import (
            nnUNetTrainerDA5 as _DA5)
        print("[DA5] nnUNetTrainerDA5 найден — сильная аугментация активна")
    except ImportError:
        print("[DA5] нет в этой версии nnunetv2 — работаю без DA5")

_BASE = _DA5 if _DA5 is not None else nnUNetTrainerNoMirroring

class nnUNetTrainerDistill(_BASE):
    def __init__(self, plans, configuration, fold, dataset_json, device="cuda:0"):
        super().__init__(plans, configuration, fold, dataset_json, device=torch.device(device))
        self.num_epochs = 1 if SMOKE else NUM_EPOCHS
        if SMOKE:
            self.num_iterations_per_epoch = 30
            self.num_val_iterations_per_epoch = 5
        else:
            self.num_val_iterations_per_epoch = 10   # pseudo-метрика: 10 итераций достаточно
        self.initial_lr = PEAK_LR                     # ⚙ peak LR (используется poly-расписанием)
        self.oversample_foreground_percent = OVERSAMPLE  # ⚙ доля форграунд-патчей
        self.save_every = 5                           # чекпоинт каждые 5 эпох (≈1 ч)
        self._t0 = None
        self._iter_times = []

    def setup_DA_params(self):
        super().setup_DA_params()
        dap = getattr(self.configuration_manager, "data_aug_params", None)
        if isinstance(dap, dict):
            dap["mirror_axes"] = None                 # студент без mirroring (в т.ч. под DA5)
            self.print_to_log_file("[DA] mirror_axes -> None")

    def configure_rotation_dummyDA_mirroring_and_inital_DA_scales(self):
        res = super().configure_rotation_dummyDA_mirroring_and_inital_DA_scales()
        # версия-независимое подавление mirroring'а (3-tuple: ... , mirror_axes)
        # сигнатура менялась между версиями: 3- или 4-tuple (mirror_axes последняя)
        if isinstance(res, tuple) and len(res) in (3, 4):
            self.print_to_log_file(f"[DA] mirror_axes {res[-1]} -> None")
            return res[:-1] + (None,)
        return res

    def on_train_start(self):
        self._t0 = time.time()
        super().on_train_start()
        # ⚙ PEAK_LR при resume: чекпоинт восстановил старый initial_lr в scheduler —
        # переприсваиваем, чтобы "продолжить с меньшим LR" реально работало
        if self.lr_scheduler is not None:
            self.lr_scheduler.initial_lr = PEAK_LR
            if hasattr(self.lr_scheduler, "max_steps"):
                self.lr_scheduler.max_steps = self.num_epochs
            self.print_to_log_file(f"[lr] peak LR = {PEAK_LR}")
        dap = getattr(self.configuration_manager, "data_aug_params", {})
        self.print_to_log_file(f"[DA] effective mirror_axes: {dap.get('mirror_axes', 'n/a')}, DA5={isinstance(self, nnUNetTrainer) and _BASE is not nnUNetTrainerNoMirroring and self.__class__ is not nnUNetTrainerNoMirroring}")
        if (os.environ.get("DISTILL_INIT", "1") == "1"
                and self.current_epoch == 0 and self.local_rank == 0):
            sd = torch.load(join(IN_RESULTS, "fold_0", "checkpoint_final.pth"),
                            map_location="cpu", weights_only=False)
            net = self.network
            if hasattr(net, "_orig_mod"):
                net = net._orig_mod
            net.load_state_dict(sd["network_weights"])  # strict
            print("[init] тёплый старт студента из fold_0: OK (strict)")
            del sd
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        # resume: долив нейтральных записей логгера (страховка от IndexError в EMA)
        if self.current_epoch > 0:
            L = self.logger.my_fantastic_logging
            n = self.current_epoch - len(L["ema_fg_dice"])
            if n > 0:
                print(f"[logger] resume: доливаю {n} записей")
                L["mean_fg_dice"].extend([0.0] * n)
                L["train_losses"].extend([0.0] * n)
                L["val_losses"].extend([0.0] * n)
                L["lrs"].extend([PEAK_LR] * n)
                L["epoch_start_timestamps"].extend([0] * n)
                L["epoch_end_timestamps"].extend([0] * n)
                L["dice_per_class_or_region"].extend([None] * n)
                v = 0.0
                for _ in range(n):
                    v = v * 0.9 + 0.1 * 0.9
                    L["ema_fg_dice"].append(v)

    def on_epoch_end(self):
        super().on_epoch_end()
        if self.local_rank == 0 and (self.current_epoch + 1) % UPLOAD_EVERY == 0:
            kaggle_ckpt_upload(join(self.output_folder, "checkpoint_latest.pth"),
                               f"epoch {self.current_epoch}")
        if self._t0 is not None and (time.time() - self._t0) > BUDGET_H * 3600:
            print(f"[budget] {BUDGET_H} ч исчерпаны — чисто завершаю на эпохе {self.current_epoch}")
            self.save_checkpoint(join(self.output_folder, "checkpoint_latest.pth"))
            kaggle_ckpt_upload(join(self.output_folder, "checkpoint_latest.pth"),
                               f"epoch {self.current_epoch} (budget stop)")
            raise _SessionOver

    def train_step(self, batch: dict) -> dict:
        if SMOKE and len(self._iter_times) < 5:
            t_iter = time.time()
        else:
            t_iter = None
        data = batch["data"]
        target = batch["target"]
        data = data.to(self.device, non_blocking=True)
        if isinstance(target, list):
            target = [i.to(self.device, non_blocking=True) for i in target]
        else:
            target = target.to(self.device, non_blocking=True)
        self.optimizer.zero_grad(set_to_none=True)
        with torch.autocast(self.device.type, enabled=self.device.type == "cuda" and not CPU_MODE):
            output = self.network(data)
            shapes = [tuple(o.shape[2:]) for o in output]
            with torch.no_grad():
                tp = teacher_probs(data)
            t_list = soft_targets_for_ds(tp, shapes)
            n = len(output)
            ws = [1.0 / (2 ** i) for i in range(n)]   # ⚙ веса DS — менять не рекомендую
            ws = [w / sum(ws) for w in ws]
            l_soft = sum(w * (W_DICE * soft_dice(o, t) + (1 - W_DICE) * soft_ce(o, t))
                         for w, o, t in zip(ws, output, t_list))
            l = l_soft
            if HARD_GT_WEIGHT > 0:
                l = (1 - HARD_GT_WEIGHT) * l_soft + HARD_GT_WEIGHT * self.loss(output, target)
        if self.grad_scaler is not None:
            self.grad_scaler.scale(l).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            l.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        if t_iter is not None:
            dt = time.time() - t_iter
            self._iter_times.append(dt)
            vram = f"{torch.cuda.memory_allocated() / 1e9:.2f} ГБ" if torch.cuda.is_available() else "cpu"
            print(f"[smoke] итерация {len(self._iter_times)}: {dt:.2f} с (VRAM {vram})")
        return {"loss": l.detach().cpu().numpy()}

In [ ]:
# ЯЧЕЙКА 11 — resume: поиск чекпоинта, защита от чужих чекпоинтов
# ============================================================
prev_ckpt = (find_one("/kaggle/input/**/nnUNetTrainerDistill__*/fold_0/checkpoint_latest.pth")
             # fallback: ЛЮБОЙ checkpoint_latest, кроме чекпоинтов УЧИТЕЛЯ
             # (у учителя fold_*/checkpoint_latest.pth имеет current_epoch≈1000!)
             or next((p for p in sorted(glob.glob("/kaggle/input/**/checkpoint_latest.pth", recursive=True))
                      if "nnUNetTrainerNoMirroring" not in p), None))
if prev_ckpt:
    print(f"[resume] найден чекпоинт прошлой сессии: {prev_ckpt}")
    # проверка содержимого перед стартом (весит <1 с):
    _sd = torch.load(prev_ckpt, map_location="cpu", weights_only=False)
    print(f"[resume]   current_epoch={_sd.get('current_epoch')}, best_ema={_sd.get('_best_ema')}, "
          f"logging={'logging' in _sd}")
    del _sd
elif REQUIRE_CKPT:
    sys.exit("[resume] чекпоинт не найден, REQUIRE_CKPT=1 — СТОП, не жгу квоту")

# nnunetv2 2.8.1: трейнер делает plans.pop("continue_training") — ключ обязателен
# (в 2.6.x лишний ключ в plans безвреден); копия, чтобы не трогать оригинал
plans_trainer = dict(plans)
plans_trainer.setdefault("continue_training", prev_ckpt is not None)
trainer = nnUNetTrainerDistill(plans_trainer, CONFIG, 0, dataset_json,
                               device="cpu" if CPU_MODE else "cuda:0")
RES_DIR = trainer.output_folder
os.makedirs(RES_DIR, exist_ok=True)
if prev_ckpt:
    shutil.copy2(prev_ckpt, join(RES_DIR, "checkpoint_latest.pth"))
    kaggle_ckpt_upload(join(RES_DIR, "checkpoint_latest.pth"),
                       "тест аплоада при старте")   # раннее обнаружение проблем с токеном

ckpt_latest = join(RES_DIR, "checkpoint_latest.pth")
need_training = True
if os.path.isfile(ckpt_latest):
    trainer.load_checkpoint(ckpt_latest)
    print(f"[resume] продолжаем с эпохи {trainer.current_epoch} "
          f"(best EMA {getattr(trainer, 'best_ema', '?')})")
    # ГВАРД: чужой/финальный чекпоинт (учителя: epoch≈1000) — не запускаем обучение
    if not SMOKE and trainer.current_epoch >= NUM_EPOCHS:
        need_training = False
        print(f"[run] эпоха {trainer.current_epoch} ≥ NUM_EPOCHS={NUM_EPOCHS} — "
              f"цикл завершён или чекпоинт чужой. Обучение пропущено.")
else:
    trainer.initialize()

In [ ]:
# ЯЧЕЙКА 12 — обучение
# ============================================================
if need_training:
    try:
        trainer.run_training()
    except _SessionOver:
        print("[run] сессия завершена по бюджету — чекпоинт сохранён")
trainer.save_checkpoint(ckpt_latest)
kaggle_ckpt_upload(ckpt_latest, f"конец сессии, эпоха {trainer.current_epoch}")
print("[run] ГОТОВО. Файлы:")
for f in glob.glob(join(RES_DIR, "*.pth")):
    print("  ", f, f"{os.path.getsize(f) / 1e6:.0f} МБ")

In [ ]:
# ЯЧЕЙКА 13 — финализация ТОЛЬКО когда цикл действительно завершён
# ============================================================
if trainer.current_epoch >= NUM_EPOCHS and not SMOKE:
    shutil.copy2(ckpt_latest, join(RES_DIR, "checkpoint_final.pth"))
    print(f"[final] цикл завершён на эпохе {trainer.current_epoch}: checkpoint_final.pth сохранён")
    # Полная sliding-window валидация (perform_actual_validation) требует raw-данных
    # в nnUNet_raw/Dataset*/imagesTr — приаттачьте их отдельным input и раскомментируйте:
    # trainer.perform_actual_validation(False)
    print("[final] числа для сравнения с учителями — только из perform_actual_validation "
          "или внешнего холдаута; pseudo-dice из лога НЕ сравним с полной валидацией")
else:
    print(f"[final] цикл не завершён (эпоха {trainer.current_epoch}/{NUM_EPOCHS}) — "
          f"продолжайте следующей сессией")

## Этап B: квантизация INT8 (NNCF) — только после завершения цикла

Восстанавливаем студента из `checkpoint_final.pth`, экспортируем ONNX →
OpenVINO IR FP32 → NNCF INT8 PTQ (калибровка на патчах валидационных томов).
Результат: `student_fp32.xml/bin` и `student_int8.xml/bin` в output сессии —
готовые модели для боевого инференса OpenVINO.

In [ ]:
# ЯЧЕЙКА 14 — экспорт IR + INT8-квантизация студента (если цикл ЗАВЕРШЁН)
# nncf 2.19.0 совместим с openvino 2025.2; ONNX opset 11
# ============================================================
CYCLE_DONE = (trainer.current_epoch >= NUM_EPOCHS) and not SMOKE
if not CYCLE_DONE:
    print(f"[quant] цикл не завершён ({trainer.current_epoch}/{NUM_EPOCHS}) — квантизация пропущена")
else:
    # ⚠ nncf НЕ тянет openvino сам (это extra) — ставим явно оба пакета
    try:
        import nncf
        import openvino  # noqa: F401
        print(f"[quant] nncf {nncf.__version__}")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "openvino", "nncf==3.4.0", "onnxscript"], check=True)
        import nncf
        import openvino  # noqa: F401

    import openvino as ov
    import blosc2

    # 1) сеть студента из checkpoint_final (без compile-обёртки)
    net_q = build_net()
    sd_q = torch.load(join(RES_DIR, "checkpoint_final.pth"), map_location="cpu", weights_only=False)
    net_q.load_state_dict(sd_q["network_weights"])
    net_q.eval()

    # 2) ONNX: шаблон [1,1,56,192,192], ось batch динамическая
    dummy = torch.randn(1, 1, 56, 192, 192)
    onnx_path = join(RES_DIR, "student_model.onnx")
    # torch >= 2.6: legacy-экспорт явно (иначе требует onnxscript/dynamo)
    export_kwargs = dict(input_names=["data"], output_names=["logits"],
                         dynamic_axes={"data": {0: "batch"}, "logits": {0: "batch"}},
                         opset_version=11)
    try:
        torch.onnx.export(net_q, dummy, onnx_path, dynamo=False, **export_kwargs)
    except TypeError:  # старый torch без параметра dynamo
        torch.onnx.export(net_q, dummy, onnx_path, **export_kwargs)
    print(f"[quant] ONNX: {onnx_path}")

    # 3) IR FP32 (эталон для сравнения FP32 vs INT8)
    ov_model = ov.convert_model(onnx_path)
    ir_fp32 = join(RES_DIR, "student_fp32.xml")
    ov.serialize(ov_model, ir_fp32)
    print(f"[quant] IR FP32: {ir_fp32}")

    # 4) калибровка: патчи 56x192x192 из первых 6 ВАЛИДАЦИОННЫХ томов (b2nd;
    #    preprocessed-томы уже нормализованы CTNormalization при подготовке)
    calib = []
    rng_c = np.random.RandomState(20261003)
    val_files = sorted(glob.glob(join(DATA_DIR, "*.b2nd")))
    val_files = [f for f in val_files if "_seg" not in f][:6]
    for vf in val_files:
        vol = np.asarray(blosc2.open(vf))
        vol = vol.squeeze(0) if vol.ndim == 4 else vol
        for _ in range(4):
            z0 = rng_c.randint(0, max(1, vol.shape[0] - 56))
            y0 = rng_c.randint(0, max(1, vol.shape[1] - 192))
            x0 = rng_c.randint(0, max(1, vol.shape[2] - 192))
            patch = vol[z0:z0+56, y0:y0+192, x0:x0+192].astype(np.float32)
            if patch.shape == (56, 192, 192):
                calib.append(patch[None, None])
    print(f"[quant] калибровка: {len(calib)} патчей")

    # NNCF 3.x: calibration_dataset = nncf.data.Dataset(source, transform_fn)
    # (в 2.x был параметр calibration_data_fn — поддержан через fallback)
    # nncf 3.x: Dataset(список_готовых_патчей) — БЕЗ transform_fn (в 3.4
    # параметр убран; локально проверено на мини-модели). Патч (1,1,z,y,x)
    # подставляется во вход "data" автоматически.
    try:
        from nncf.data import Dataset as NncfDataset
        nncf_ds = NncfDataset(calib)
        quant_model = nncf.quantize(ov_model, nncf_ds,
                                    preset=nncf.QuantizationPreset.MIXED,
                                    subset_size=len(calib))
    except TypeError:
        # старый API nncf 2.x
        quant_model = nncf.quantize(ov_model,
                                    calibration_data_fn=lambda: iter(calib),
                                    preset=nncf.QuantizationPreset.MIXED,
                                    subset_size=len(calib))
    ir_int8 = join(RES_DIR, "student_int8.xml")
    ov.serialize(quant_model, ir_int8)
    print(f"[quant] IR INT8: {ir_int8}")
    print("[quant] ГОТОВО: student_fp32.xml/bin + student_int8.xml/bin в output")


## Этап C: валидация INT8-студента на val-сплите (Kaggle-шкала)

Sliding-window инференс INT8-модели на 5 val-кейсах + Dice по органам.
⚠ Эти числа — **внутренняя (Kaggle) шкала**: сплит контаминирован учителями
(они обучались на этих же кейсах), Dice завышён. Прод-гейт считается
**локально на внешнем холдауте** (кейсы, не виденные учителями).

In [ ]:
# ЯЧЕЙКА 15 — sliding-window валидация INT8 + Dice по органам (5 кейсов)
# Упрощённый движок (равномерные веса, без crop тела): для валидации достаточно
# ============================================================
if not CYCLE_DONE:
    print("[val] цикл не завершён — валидация пропущена")
else:
    # openvino <= 2025: подпакет runtime; openvino >= 2026: сам openvino
    try:
        import openvino.runtime as ovrt
    except ModuleNotFoundError:
        import openvino as ovrt
    import blosc2

    core_v = ovrt.Core()
    comp_v = core_v.compile_model(ir_int8, "CPU")
    req_v = comp_v.create_infer_request()
    in_name = comp_v.input(0).any_name
    out_name = comp_v.output(0).any_name

    PZ, PY, PX = 56, 192, 192
    STRIDE = 0.5

    def infer_volume(vol):
        steps = []
        for d, p in zip(vol.shape, (PZ, PY, PX)):
            if d < p:
                steps.append([0])
            else:
                s = max(1, int(p * STRIDE))
                st = list(range(0, d - p + 1, s))
                if not st or st[-1] + p < d:
                    st.append(d - p)
                steps.append(st)
        logits = np.zeros((32,) + vol.shape, dtype=np.float32)
        wsum = np.zeros(vol.shape, dtype=np.float32)
        for z0 in steps[0]:
            for y0 in steps[1]:
                for x0 in steps[2]:
                    patch = vol[z0:z0+PZ, y0:y0+PY, x0:x0+PX]
                    # infer() -> dict{имя_выхода: тензор} (get_output_tensor()
                    # не работает на модели с dynamic-batch выходом)
                    res = req_v.infer({in_name: patch[None, None]})
                    out = res[out_name]
                    logits[:, z0:z0+PZ, y0:y0+PY, x0:x0+PX] += out[0]
                    wsum[z0:z0+PZ, y0:y0+PY, x0:x0+PX] += 1
        logits /= np.maximum(wsum, 1e-8)
        return np.argmax(logits, axis=0).astype(np.uint8)

    def dice(a, b):
        a, b = a.astype(bool), b.astype(bool)
        denom = int(a.sum()) + int(b.sum())
        if denom == 0:
            return None
        return 2.0 * (a & b).sum() / denom

    for case in val_ids[:5]:
        vf = join(DATA_DIR, case + ".b2nd")
        sf = join(DATA_DIR, case + "_seg.b2nd")
        vol = np.asarray(blosc2.open(vf))
        vol = vol.squeeze(0) if vol.ndim == 4 else vol
        gt = np.asarray(blosc2.open(sf))
        gt = gt.squeeze(0) if gt.ndim == 4 else gt
        seg = infer_volume(vol.astype(np.float32))
        dices = {}
        for name, lbl in dataset_json["labels"].items():
            if lbl == 0:
                continue
            d = dice(seg == lbl, gt == lbl)
            if d is not None:
                dices[name] = float(d)
        m = np.mean(list(dices.values()))
        print(f"[val] {case}: mean Dice = {m:.4f} ({len(dices)} органов)")
    print("[val] напоминание: сплит контаминирован учителями — честные числа "
          "только на внешнем холдауте (локально)")
